# IndoML T2 — Is the test set harder than validation? (analysis only, ~40 min, Kaggle T4)
Estimated SNR (kept speech vs removed noise by D), noise occupancy, loudness, length, clipping — TEST (800 clips) vs VALIDATION synthetic/natural (300 each).
Inputs: `indoml-validation`, `indoml-t2-ckpt`, `mpsenet-e10r`, `indoml-track2-test`. GPU T4, Internet ON, HF_TOKEN ON.

In [ ]:
!rm -rf /kaggle/working/ner && git clone -q --depth 1 https://github.com/nagshubhadip/noise-event-detection-and-removal /kaggle/working/ner
!pip -q install jiwer sentencepiece webrtcvad-wheels librosa 2>&1 | tail -1
!wget -q https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip -O /kaggle/working/esc50.zip && unzip -q -o /kaggle/working/esc50.zip -d /kaggle/working/ && ls /kaggle/working/ESC-50-master | head
# ---- MP-SENet (MIT) ----
!rm -rf /kaggle/working/mpsenet && git clone -q --depth 1 https://github.com/yxlu-0102/MP-SENet /kaggle/working/mpsenet
!pip -q install einops 2>&1 | tail -1

In [ ]:
import sys, os, glob, json, time, random, numpy as np, torch, soundfile as sf, librosa
from pathlib import Path
VAL = Path('/kaggle/input/datasets/avin55/indoml-validation/validation')
M = Path('/kaggle/working/ner/track2_removal/versions/v-1_indoml2026_track2/modular')
d = (M/'data.py').read_text()
d = d.replace('Path("/kaggle/input/indoml-validation"),', 'Path("/kaggle/input/datasets/avin55/indoml-validation/validation"), Path("/kaggle/input/indoml-validation"),')
d = d.replace('cid = (r.get("clip_id") or r.get("id")', 'cid = (r.get("segmentFileName") or r.get("clip_id") or r.get("id")')
(M/'data.py').write_text(d); sys.path.insert(0, str(M))
from kaggle_secrets import UserSecretsClient
HF = UserSecretsClient().get_secret("HF_TOKEN"); os.environ["HF_TOKEN"] = HF
import config, data, evaluate
import model as model_mod
from train import train_on_sim
from inference import enhance, Transcriber
from jiwer import wer as jwer
config.ensure_dirs()
val_dir, items = data.resolve_validation()
meta = {m['segmentFileName'][:-4]: m for m in json.load(open(VAL/'validationMetadata.json', encoding='utf-8'))}
print('validation clips', len(items))
import torch.nn.functional as F

In [ ]:
import types, sys
sys.modules.setdefault('pesq', types.SimpleNamespace(pesq=lambda *a, **k: 0.0))   # PESQ not needed (no metric discriminator)
sys.path.insert(0, '/kaggle/working/mpsenet')
from env import AttrDict
from dataset import mag_pha_stft, mag_pha_istft
from models.model import MPNet, phase_losses
h = AttrDict(json.load(open('/kaggle/working/mpsenet/best_ckpt/config.json')))
dev = torch.device('cuda')
def load_mp(path='/kaggle/working/mpsenet/best_ckpt/g_best_dns'):
    m = MPNet(h).to(dev); m.load_state_dict(torch.load(path, map_location=dev, weights_only=False)['generator']); return m.eval()
def mp_forward(m, wav):                       # wav [B, T] (RMS-normalised) -> enhanced [B, T]
    amp, pha, _ = mag_pha_stft(wav, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
    amp_g, pha_g, com_g = m(amp, pha)
    return mag_pha_istft(amp_g, pha_g, h.n_fft, h.hop_size, h.win_size, h.compress_factor), (amp_g, pha_g, com_g)
@torch.no_grad()
def enhance_mp(m, x, win=4.0, hop=3.0):
    x = np.asarray(x, np.float32); n = len(x); sr = 16000
    nf = float(np.sqrt(n / (np.sum(x**2) + 1e-9))); xn = torch.from_numpy(x * nf).to(dev)
    W, H = int(win * sr), int(hop * sr)
    if n <= W:
        y = mp_forward(m, xn[None])[0][0][:n]
    else:
        y = torch.zeros(n, device=dev); wsum = torch.zeros(n, device=dev)
        fade = torch.ones(W, device=dev); r = W - H; fade[:r] = torch.linspace(0, 1, r, device=dev); fade[-r:] = torch.linspace(1, 0, r, device=dev)
        s = 0
        while True:
            e = min(s + W, n); seg = xn[s:e]
            if len(seg) < W: seg = torch.nn.functional.pad(seg, (0, W - len(seg)))
            out = mp_forward(m, seg[None])[0][0][:W]
            L = e - s; f = fade[:L].clone()
            if s == 0: f[:r] = 1
            if e == n: f[-min(r, L):] = 1
            y[s:e] += out[:L] * f; wsum[s:e] += f
            if e == n: break
            s += H
        y = y / wsum.clamp_min(1e-6)
    y = (y / nf).cpu().numpy()
    return np.pad(y, (0, max(0, n - len(y))))[:n].astype(np.float32)
mp = load_mp(); print('MP-SENet params (M):', round(sum(p.numel() for p in mp.parameters()) / 1e6, 2))

In [ ]:
# Isolated validation noise events + fingerprint clustering
SR = 16000
NV = Path('/kaggle/working/bank/val_noise'); NV.mkdir(parents=True, exist_ok=True)
segs, fps, owner = [], [], []
for v in items:
    if not v['syn']: continue
    y = data.read_audio(v['path']); c = data.read_audio(v['ref']); L = min(len(y), len(c)); n = (y[:L] - c[:L]).astype(np.float32)
    for k, e in enumerate(meta.get(v['id'], {}).get('NoiseSubCategoryTimeStamp', [])):
        a, b = max(0, int((float(e['start']) - 0.02) * SR)), min(L, int((float(e['end']) + 0.02) * SR))
        if b - a < int(0.3 * SR) or np.abs(n[a:b]).max() < 1e-4: continue
        seg = n[a:b]; p = NV / f"{v['id'][-40:]}_{k}.wav"; sf.write(p, seg, SR)
        mel = librosa.power_to_db(librosa.feature.melspectrogram(y=seg, sr=SR, n_mels=48, n_fft=512, hop_length=160) + 1e-10)
        f = librosa.util.fix_length(mel, size=48, axis=1).ravel(); f = (f - f.mean()) / (f.std() + 1e-6)
        segs.append({'path': str(p), 'dur': (b - a) / SR}); fps.append(f / np.linalg.norm(f)); owner.append(v['id'])
F = np.stack(fps); cents, lab = [], np.zeros(len(F), int)
for i, f in enumerate(F):
    if cents:
        s = np.stack(cents) @ f; j = int(s.argmax())
        if s[j] > 0.95: lab[i] = j; continue
    cents.append(f); lab[i] = len(cents) - 1
ncl = len(cents); rng = np.random.default_rng(7)
print('noise events', len(F), '| clusters', ncl)
for frac in [0.35, 0.45, 0.55]:
    held = set(rng.permutation(ncl)[:int(frac * ncl)].tolist())
    clip_cl = {}
    for o, l in zip(owner, lab): clip_cl.setdefault(o, set()).add(int(l))
    HELD_CLIPS = [o for o, s in clip_cl.items() if s <= held]
    if len(HELD_CLIPS) >= 80: break
train_noise = [s for s, l in zip(segs, lab) if int(l) not in held]
print(f'held-out clusters {len(held)}/{ncl} | held-out eval clips {len(HELD_CLIPS)} | train noise segments {len(train_noise)}')

In [ ]:
# Load model D (E10-R annealed) from the uploaded bundle dataset; rebuild the .pt if Kaggle extracted it
import zipfile, gc, torch.nn.functional as F
def find_ckpt(key):
    plain = [x for x in glob.glob(f'/kaggle/input/**/*{key}*.pt', recursive=True) if os.path.isfile(x)]
    if plain: return plain[0]
    roots = [Path(x).parent for x in glob.glob('/kaggle/input/**/data.pkl', recursive=True) if key in x]
    assert roots, f'{key} not found: attach the mpsenet-e10r dataset (all_models_bundle.zip)'
    out = Path(f'/kaggle/working/{key}_rebuilt.pt')
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_STORED) as z:
        for f in sorted(roots[0].rglob('*')):
            if f.is_file(): z.write(f, f'{roots[0].name}/{f.relative_to(roots[0]).as_posix()}')
    return str(out)
D_CK = find_ckpt('anneal'); B_CK = find_ckpt('warm_e8')
mD = load_mp(D_CK); print('D:', D_CK, '| B:', B_CK)
del mp; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# DIAGNOSTIC: is TEST harder than VALIDATION? Compare estimated SNR / noise occupancy / loudness / length (analysis only, no gating)
test_paths = sorted(x for x in glob.glob('/kaggle/input/**/indoml-track2-test/audio/*.wav', recursive=True))
rng = np.random.default_rng(3); TP = [test_paths[j] for j in rng.permutation(len(test_paths))[:800]]
VS = [v['path'] for v in items if v['syn']]; VN = [v['path'] for v in items if not v['syn']]
VS = [VS[j] for j in rng.permutation(len(VS))[:300]]; VN = [VN[j] for j in rng.permutation(len(VN))[:300]]
def stats(paths):
    out = []
    for p in paths:
        x = data.read_audio(p); s = enhance_mp(mD, x, win=8.0, hop=6.0); n = x - s
        es, en = float((s**2).sum()) + 1e-12, float((n**2).sum()) + 1e-12
        fr = 320; k = len(x) // fr
        if k == 0: continue
        ef_n = (n[:k*fr].reshape(k, fr)**2).mean(1); ef_s = (s[:k*fr].reshape(k, fr)**2).mean(1)
        occ = float(np.mean(ef_n > ef_s))                       # share of 20 ms frames where removed noise > kept speech
        loud = float(np.mean(ef_n > 0.1 * ef_s.max()))           # share of frames with clearly audible noise
        out.append(dict(snr=10*np.log10(es/en), occ=occ, loud=loud, rms=float(np.sqrt((x**2).mean())), dur=len(x)/16000, clip=float(np.mean(np.abs(x) > 0.99))))
    return out
t0 = time.time(); R = {'TEST (800)': stats(TP), 'VAL synthetic (300)': stats(VS), 'VAL natural (300)': stats(VN)}; print(f'{(time.time()-t0)/60:.1f} min')
def q(a): a = np.array(a); return f'p10 {np.percentile(a,10):6.2f} | p50 {np.percentile(a,50):6.2f} | p90 {np.percentile(a,90):6.2f}'
for key in ['snr', 'occ', 'loud', 'rms', 'dur', 'clip']:
    print(f'--- {key}')
    for name, rows in R.items(): print(f'  {name:22s}', q([r[key] for r in rows]))
# share of clips whose estimated SNR is very low
for name, rows in R.items():
    a = np.array([r['snr'] for r in rows]); print(f'{name:22s} clips with est. SNR < 0 dB: {np.mean(a < 0):.0%} | < 5 dB: {np.mean(a < 5):.0%}')
json.dump({k: v for k, v in R.items()}, open('/kaggle/working/test_vs_val_stats.json', 'w'))